# Template 2: final CUDA + Mac candidate merge
Run on a CPU runtime. The four uploaded ZIPs in My Drive/Template2-Colab are checksum-verified. All original results remain intact. Numerical success does not imply semantic quality approval; flagged masks and the fixed review sample remain required.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/Template2-Colab')
assert DRIVE.is_dir(), 'Create My Drive/Template2-Colab and upload the two ZIP files.'


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile, time
ROOT = Path('/content/template2_project')
ROOT.mkdir(exist_ok=True)
def digest(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
def safe_extract(path, root):
    with zipfile.ZipFile(path) as z:
        for member in z.infolist():
            assert (root/member.filename).resolve().is_relative_to(root.resolve()), 'Unsafe archive path'
            assert (member.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink rejected'
        z.extractall(root)
EXPECTED = {'template2_colab_code.zip': {'bytes': 1982989, 'sha256': '26c6151d18c49fc17735c3d53f9406c45e91273e6e3a44560fabab4f260d7f18'}, 'template2_imagenet10_sources.zip': {'bytes': 1631349237, 'sha256': '6af6b40f1399a874cf40441169168d891ff8a6d0a13dd3a60154de26dd4db14e'}}

for name, info in EXPECTED.items():
    source=DRIVE/name
    deadline=time.monotonic()+3600
    while not source.exists():
        if time.monotonic()>deadline: raise TimeoutError(f'Upload still missing after one hour: {source}. Finish the Drive upload and rerun this cell.')
        print(f'Waiting for Drive upload: {name}',flush=True)
        time.sleep(30)
    local=Path('/content')/name
    if not local.exists() or digest(local)!=info['sha256']:
        shutil.copyfile(source,local)
    assert local.stat().st_size==info['bytes'] and digest(local)==info['sha256'], f'Bundle damaged: {name}'
    safe_extract(local,ROOT)
print('Both bundles verified and extracted to local VM storage.')


In [ ]:
import subprocess,sys
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','uv==0.8.22'],check=True)
UV=[sys.executable,'-m','uv'];ENV=Path('/content/template2_audit_env');PY=str(ENV/'bin/python')
if not Path(PY).exists():subprocess.run(UV+['venv','--python','3.11',str(ENV)],check=True)
subprocess.run(UV+['pip','install','--python',PY,'numpy==2.4.6','pillow==12.3.0'],check=True)
print('CPU audit environment ready; no model inference or GPU needed.')


In [ ]:
# Run after mounting Drive, extracting original bundles, and creating PY (Python 3.11 + pinned NumPy/Pillow).
from pathlib import Path
import os,sys,hashlib,shutil,subprocess,time
DRIVE=Path('/content/drive/MyDrive/Template2-Colab');CUDA=Path('/content/template2_project')
MAC=Path('/content/template2_mac_remainder');MERGED=Path('/content/template2_merged');SUPPORT=Path('/content/template2_merge_support')
assert DRIVE.is_dir(), 'Mount the project Drive folder first'
sys.path.insert(0,str(CUDA))
from colab_checkpoint import extract,restore
restore(DRIVE/'checkpoints',CUDA)
for name,checksum,dest in [
 ('template2_mac_remainder.zip','8621c14c92659a3fc816e0b422bd91451d210e6ef43172d9ac98612f65853217',MAC),
 ('template2_merge_code.zip','a7b6bbcdaf8907cba178ff08a355c91f08bb0ed1579459ce2de9a4ac00f341f1',SUPPORT)]:
 source=DRIVE/name;deadline=time.monotonic()+1800
 while not source.exists():
  if time.monotonic()>deadline:raise TimeoutError('Complete Drive upload: '+name)
  print('Waiting for completed Drive upload:',name,flush=True);time.sleep(30)
 local=Path('/content')/name;shutil.copyfile(source,local)
 with local.open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()==checksum, 'Archive hash mismatch'
 dest.mkdir(exist_ok=True);extract(local,dest);print('Verified and extracted',name,flush=True)
env={**os.environ,'TEMPLATE2_BACKUP_DIR':str(DRIVE/'merged_checkpoints')}
report=MERGED/'reports/background_bias_current/production_v3_merged_cuda_mps';report.mkdir(parents=True,exist_ok=True)
with (report/'merge.log').open('a',buffering=1) as log:
 worker=subprocess.Popen([PY,'-u',str(SUPPORT/'merge_results.py'),str(CUDA),str(MAC),str(MERGED)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
 for line in worker.stdout:print(line,end='',flush=True);log.write(line)
 if worker.wait():raise RuntimeError('Merge/audit failed; original inputs remain intact. Inspect the output above.')
print('Results saved to:',DRIVE/'merged_checkpoints/production_v3_merged_cuda_mps',flush=True)
